# Libraries

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.pardir))

import pandas as pd
import mlflow
from src import data_processing
from src import model_building

# Preparing Data and Model

In [2]:
dataset_path = os.path.join('..','datasets','Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)
train, test = data_processing.process_data(
    df = df, 
    feature_col = 'Content',
    label_col = 'Label',
    random_state = 42,
    df_name = 'Philippine Fake News Corpus.csv'
)

In [5]:
trainloader, testloader = model_building.create_dataloaders(train, test, 'Content', 'Label')

In [6]:
trainloader, testloader

(<torch.utils.data.dataloader.DataLoader at 0x13a8f6e46e0>,
 <torch.utils.data.dataloader.DataLoader at 0x13a9098e7b0>)

## Verify Dataset

In [7]:
for batch in trainloader:
    sample_train = batch
    break

for batch in testloader:
    sample_test = batch
    break

print(f'x_train shape: {sample_train[0].shape} | x_test shape: {sample_test[0].shape}')
print(f'x_train dtype: {sample_train[0].dtype} | x_test shape: {sample_test[0].dtype}\n')

print(f'y_train shape: {sample_train[1].shape} | y_test shape: {sample_test[1].shape}')
print(f'y_train dtype: {sample_train[1].dtype} | y_test dtype: {sample_test[1].dtype}')

x_train shape: torch.Size([32, 24186]) | x_test shape: torch.Size([32, 24186])
x_train dtype: torch.int64 | x_test shape: torch.int64

y_train shape: torch.Size([32]) | y_test shape: torch.Size([32])
y_train dtype: torch.int64 | y_test dtype: torch.int64


In [8]:
max_seq = sample_train[0].size(1)
max_seq

24186

In [9]:
expectations = {
    'Similar shapes': (
        (sample_train[0].shape == sample_test[0].shape) &
        (sample_train[1].shape == sample_test[1].shape)
    ),
    'Similar dtypes': (
        (sample_train[0].dtype == sample_test[0].dtype) &
        (sample_train[1].dtype == sample_test[1].dtype)
    )
}

expectations

{'Similar shapes': True, 'Similar dtypes': True}

# Build Model

In [10]:
config = {
    'vocab_size': 8000,
    'embed_dim': 5,
    'pad_id': 3,
    'conv_dim': 4,
    'kernel_size': 5,
    'max_seq': max_seq,
}

model = model_building.FakeNewsDetector(**config)

In [11]:
model_dir = os.path.join('..','models','FakeNewsDetector')
os.listdir(model_dir)

['225bddb08550475ba1bd78085bf05958',
 '4d2ed0343c554989a322540d8fc060bc',
 '859f1744b6ac44c8ad22f79ab61b658b',
 'e2f7a87641c748b99899751bbf1a8ce9',
 'f488585343114bbb87add6fd998d544a',
 'mlflow.db']

## Verifiying if its the same model as the logged models

In [12]:
model_building.get_experiment(
    exp_name = 'FakeNewsDetector',
    uri_path = os.path.join(model_dir, 'mlflow.db')
)

<Experiment: artifact_location=('file:///d:/zPersonal/Tools/VS Code/VSCode Script '
 'Folder/GithubRepoTemps/FakeNewsDetection/notebooks/../models/FakeNewsDetector'), creation_time=1779880707986, experiment_id='1', last_update_time=1779880707986, lifecycle_stage='active', name='FakeNewsDetector', tags={}, trace_location=None, workspace='default'>

In [13]:
recent_runs = mlflow.search_runs(
    filter_string = "status = 'FINISHED'",
    order_by = ['end_time DESC', 'metrics.test_loss ASC', 'metrics.train_loss ASC'],
    search_all_experiments = True
)
recent_runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.train_loss,metrics.test_loss,metrics.Loss,params.embed_dim,params.max_seq,params.vocab_size,params.kernel_size,params.pad_id,params.conv_dim,tags.mlflow.user,tags.version,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name
0,e2f7a87641c748b99899751bbf1a8ce9,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-06-02 09:21:53.360000+00:00,2026-06-02 09:43:58.695000+00:00,0.000015,0.002698,NaN,5,24186,8000,5,3,4,kayle,1.0,NOTEBOOK,tasteful-fawn-292,model_building.ipynb
1,4d2ed0343c554989a322540d8fc060bc,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-06-01 10:01:19.649000+00:00,2026-06-01 10:29:53.726000+00:00,0.000010,0.003674,NaN,5,24186,8000,5,3,4,kayle,1.0,NOTEBOOK,bold-moth-290,model_building.ipynb
2,f488585343114bbb87add6fd998d544a,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 09:41:40.357000+00:00,2026-05-30 09:48:27.933000+00:00,NaN,NaN,0.010168,5,24186,8000,5,3,4,kayle,1.0,NOTEBOOK,charming-cat-820,model_building.ipynb
3,859f1744b6ac44c8ad22f79ab61b658b,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-30 08:46:25.649000+00:00,2026-05-30 08:53:31.161000+00:00,NaN,NaN,0.075763,5,24186,8000,5,3,4,kayle,None,NOTEBOOK,sincere-cat-696,model_building.ipynb
4,225bddb08550475ba1bd78085bf05958,1,FINISHED,file:///d:/zPersonal/Tools/VS Code/VSCode Scri...,2026-05-28 10:55:44.448000+00:00,2026-05-28 11:01:53.473000+00:00,NaN,NaN,2.548278,5,24186,8000,5,3,4,kayle,None,NOTEBOOK,caring-bee-173,model_building.ipynb


In [14]:
latest_run = recent_runs.loc[0, :]
latest_run

run_id                                      e2f7a87641c748b99899751bbf1a8ce9
experiment_id                                                              1
status                                                              FINISHED
artifact_uri               file:///d:/zPersonal/Tools/VS Code/VSCode Scri...
start_time                                  2026-06-02 09:21:53.360000+00:00
end_time                                    2026-06-02 09:43:58.695000+00:00
metrics.train_loss                                                  0.000015
metrics.test_loss                                                   0.002698
metrics.Loss                                                             NaN
params.embed_dim                                                           5
params.max_seq                                                         24186
params.vocab_size                                                       8000
params.kernel_size                                                         5

In [15]:
expected_params = {
    'vocab_size': int(latest_run['params.vocab_size']),
    'embed_dim': int(latest_run['params.embed_dim']),
    'pad_id': int(latest_run['params.pad_id']),
    'conv_dim': int(latest_run['params.conv_dim']),
    'kernel_size': int(latest_run['params.kernel_size']),
    'max_seq': int(latest_run['params.max_seq'])
}

for k,v in config.items():
    print(f'Same {k}: {v == expected_params[k]}')

Same vocab_size: True
Same embed_dim: True
Same pad_id: True
Same conv_dim: True
Same kernel_size: True
Same max_seq: True


## Load Latest Model 

In [16]:
uri_path = os.path.join(model_dir, 'model.db')

model_building.load_latest_model(model, model_dir)

# Command Line Tool Tests

In [6]:
import argparse

In [27]:
parser = argparse.ArgumentParser()
parser.add_argument('-n', '--name', help = 'Your Name')

_StoreAction(option_strings=['-n', '--name'], dest='name', nargs=None, const=None, default=None, type=None, choices=None, required=False, help='Your Name', metavar=None)

In [30]:
args = parser.parse_args(args = ['-n Test'])

print(args.name)

 Test


In [88]:
parser = argparse.ArgumentParser(description = 'Train a Filipino Fake News Detector')
parser.add_argument('-e', '--epochs', type = int, help = 'Number of times the model sees the whole dataset in training')
parser.add_argument('-ed', '--embed_dim', type = int, default = 5, help = 'Number of embedding dimensions for the model')
parser.add_argument('-cd', '--conv_dim', type = int, default = 4, help = 'Number of convolutional dimensions for the model')
parser.add_argument('-k', '--kernel_size', type = int, default = 5, help = 'Kernel size of the Convolutional and Pooling Layers')
parser.add_argument('-vc', '--vocab_size', type = int, default = 8000, help = 'Size of the vocabulary the model is trained on')
parser.add_argument('-pid', '--pad_id', type = int, default = 3, help = 'Integer used to register as the padding token')
parser.add_argument('-eid', '--eos_id', type = int, default = 2, help = 'Integer used to register as the end-of-sequence token')
parser.add_argument('-b', '--batch_size', type = int, default = 32, help = 'Size of the batches for the dataloader the model uses')
parser.add_argument('-v', '--verbose', type = bool, default = True, help = 'Whether or not the program should output progress reports')
parser.add_argument('-rs', '--random_state', type = int, default = 42, help = 'Seed used for the psuedo-random functions in the program')

_StoreAction(option_strings=['-rs', '--random_state'], dest='random_state', nargs=None, const=None, default=42, type=<class 'int'>, choices=None, required=False, help='Seed used for the psuedo-random functions in the program', metavar=None)

In [89]:
args = parser.parse_args(args = ['-e5']) # args[] is used since this is how it works in notebooks
print(args)

Namespace(epochs=5, embed_dim=5, conv_dim=4, kernel_size=5, vocab_size=8000, pad_id=3, eos_id=2, batch_size=32, verbose=True, random_state=42)


In [90]:
args = parser.parse_args(args = ['--epochs', '5'])
print(args)

Namespace(epochs=5, embed_dim=5, conv_dim=4, kernel_size=5, vocab_size=8000, pad_id=3, eos_id=2, batch_size=32, verbose=True, random_state=42)


In [91]:
args = parser.parse_args(args = ['--epochs=5'])
print(args)

Namespace(epochs=5, embed_dim=5, conv_dim=4, kernel_size=5, vocab_size=8000, pad_id=3, eos_id=2, batch_size=32, verbose=True, random_state=42)


In [ ]:
config = {
    'vocab_size': args.vocab_size,
    'embed_dim': args.embed_dim,
    'pad_id': args.pad_id,
    'conv_dim': args.conv_dim,
    'kernel_size': args.kernel_size,
    'max_seq': args.max_seq,
}